# 🚀 Multimodal RAG for Spaceflight Procedures — Live Demo
**BPC Data Engineering SS26**

This notebook demonstrates three retrieval approaches on NASA ISS emergency procedures:

| Track | Approach | What it uses |
|-------|----------|--------------|
| **Track 1** | Raw Text Vector | Plain markdown chunks |
| **Track 2** | Enriched Text Vector | Markdown + injected image captions |
| **Track 3** | KG Structured Search | Neo4j graph traversal |
| **Track 4** | Hybrid | Vector seeds → graph expansion |

---
## 0. Setup

In [ ]:
import sys, os
from pathlib import Path
from IPython.display import display, Markdown, Image as IPImage

# Make sure src/ is on the path
sys.path.insert(0, str(Path("../src").resolve()))

# Load env
from dotenv import load_dotenv
load_dotenv("../config/.env")

# LLM client for answer generation
from openai import OpenAI
client = OpenAI(
    api_key=os.getenv("SAIA_API_KEY"),
    base_url=os.getenv("SAIA_BASE_URL"),
    timeout=60,
)
MODEL = os.getenv("SAIA_DEFAULT_MODEL")
print(f"✅ Model: {MODEL}")

ANSWER_PROMPT = """You are a medical assistant for NASA ISS spaceflight emergency procedures.
Answer the question using ONLY the provided context. Be concise and step-oriented.
If the context is insufficient, say so explicitly."""

def generate_answer(question: str, context_chunks: list) -> str:
    if not context_chunks:
        return "⚠️ No context retrieved."
    parts = []
    for i, c in enumerate(context_chunks):
        text = c.get("text") or c.get("chunk_text") or c.get("step_text") or ""
        doc  = c.get("doc", "unknown")
        if text:
            parts.append(f"[{i+1}] (from {doc})\n{text.strip()}")
    context_str = "\n\n".join(parts) or "No context."
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ANSWER_PROMPT},
            {"role": "user",   "content": f"Context:\n{context_str}\n\nQuestion: {question}"},
        ],
        temperature=0.1, max_tokens=600,
    )
    return resp.choices[0].message.content.strip()

def show_result(question, answer, context_chunks, track_name):
    """Pretty-prints a retrieval result for the demo."""
    display(Markdown(f"### {track_name}"))
    display(Markdown(f"**❓ Question:** {question}"))
    display(Markdown(f"**💬 Answer:**\n\n{answer}"))
    display(Markdown(f"**📄 Sources retrieved:** {len(context_chunks)} chunk(s)"))
    docs = list({c.get('doc','?') for c in context_chunks if c.get('doc')})
    display(Markdown(f"**📁 From documents:** `{'`, `'.join(docs)}`"))
    print("─" * 70)

---
## 1. Track 1 — Raw Text Vector Baseline
Plain markdown chunks, no image information at all.

In [ ]:
from retrieval.search_text import retrieve_text_context

# A straightforward procedural question — text retrieval should handle this well
Q1 = "What are the steps to deploy an EpiPen during a severe allergic reaction?"

context_t1 = retrieve_text_context(query=Q1, top_k=4)
answer_t1  = generate_answer(Q1, context_t1)
show_result(Q1, answer_t1, context_t1, "Track 1 — Raw Text")

In [ ]:
# Show the raw chunks that were retrieved
display(Markdown("#### Retrieved chunks (Track 1)"))
for i, c in enumerate(context_t1):
    display(Markdown(f"**Chunk {i+1}** — `{c.get('doc')}`"))
    print(c.get('text','')[:300])
    print()

---
## 2. Track 2 — Enriched Text Vector
Same chunks but the markdown has been enriched with LLM-generated image captions and OCR text.

> **Key demo point:** Ask a question that requires understanding what is *shown in a figure*.
> Track 1 will miss this; Track 2 should answer it because the caption is now searchable text.

In [ ]:
from retrieval.search_enriched import retrieve_enriched_context

# ── Image-dependent question ──────────────────────────────────────────────────
# This question can only be answered if the figure caption/OCR was injected.
# Track 1 will return generic text; Track 2 should return the figure description.
Q2_IMAGE = "Where exactly should the AED electrodes be placed on the patient's chest?"

context_t2_img = retrieve_enriched_context(query=Q2_IMAGE, top_k=4)
answer_t2_img  = generate_answer(Q2_IMAGE, context_t2_img)
show_result(Q2_IMAGE, answer_t2_img, context_t2_img, "Track 2 — Enriched (image question)")

In [ ]:
# ── Side-by-side comparison: same question through Track 1 vs Track 2 ─────────
display(Markdown("### Side-by-side: Track 1 vs Track 2 on the same image question"))

context_t1_img = retrieve_text_context(query=Q2_IMAGE, top_k=4)
answer_t1_img  = generate_answer(Q2_IMAGE, context_t1_img)

display(Markdown("#### Track 1 answer (no image info):"))
display(Markdown(answer_t1_img))

display(Markdown("#### Track 2 answer (with image caption injected):"))
display(Markdown(answer_t2_img))

In [ ]:
# ── Show the actual figure the caption came from ──────────────────────────────
display(Markdown("#### The figure that Track 2 can now 'see':"))

fig_path = Path("../data/images/1.102_AED_ASSISTED_CPR/fig1.png")
if fig_path.exists():
    display(IPImage(str(fig_path), width=420))
    display(Markdown("*Figure 1 — AED Electrode Placement (from 1.102 AED ASSISTED CPR)*"))
else:
    print(f"Image not found at {fig_path} — adjust path if needed.")

In [ ]:
# ── Second image-dependent question ──────────────────────────────────────────
Q2_IMAGE2 = "What does the correct hand position look like when activating the intraosseous device?"

context_t2_b = retrieve_enriched_context(query=Q2_IMAGE2, top_k=4)
answer_t2_b  = generate_answer(Q2_IMAGE2, context_t2_b)
show_result(Q2_IMAGE2, answer_t2_b, context_t2_b, "Track 2 — Enriched (second image question)")

# Show the figure if it exists
fig2 = Path("../data/images/1.103_ALS_ALGORITHM/fig4.png")
if fig2.exists():
    display(IPImage(str(fig2), width=380))
    display(Markdown("*Figure 4 — Activating Intraosseous Device (from 1.103 ALS ALGORITHM)*"))

In [ ]:
# ── Show what was actually injected into the enriched chunk ───────────────────
display(Markdown("#### What Track 2 retrieved (enriched chunk with injected caption):"))
for i, c in enumerate(context_t2_img):
    text = c.get('text', '')
    display(Markdown(f"**Chunk {i+1}** — `{c.get('doc')}`"))
    # Highlight the injected IMAGE DESCRIPTION block if present
    if '[IMAGE DESCRIPTION:' in text:
        start = text.index('[IMAGE DESCRIPTION:')
        print("...")
        print(text[max(0, start-100):start+300])
    else:
        print(text[:300])
    print()

---
## 3. Track 3 — KG Structured Search
Deterministic Cypher query over Neo4j. No embeddings — pure entity matching.
Returns Steps, linked Figures, and Warnings directly from the graph.

In [ ]:
from retrieval.search_kg import retrieve_kg_context

# Entity-specific question — the KG excels here because
# it matches exact terms against Step nodes
Q3_ENTITIES = ["Epinephrine", "ALS"]
Q3 = "How is Epinephrine administered using the ALS intraosseous needle?"

context_t3 = retrieve_kg_context(query_entities=Q3_ENTITIES)
answer_t3  = generate_answer(Q3, context_t3)
show_result(Q3, answer_t3, context_t3, "Track 3 — KG Structured Search")

In [ ]:
# Show the raw KG rows — this demonstrates the structured nature of the retrieval
display(Markdown("#### Raw KG rows returned by Cypher:"))
for i, row in enumerate(context_t3[:5]):
    display(Markdown(f"**Row {i+1}**"))
    print(f"  Step {row.get('step_number')}: {row.get('step_text')}")
    print(f"  Doc        : {row.get('doc')}")
    if row.get('llm_caption'):
        print(f"  Fig caption: {row.get('llm_caption')[:100]}")
    if row.get('warning'):
        print(f"  ⚠️  Warning : {row.get('warning')[:100]}")
    print()

In [ ]:
# ── Warning-specific question — unique KG advantage ──────────────────────────
Q3_WARN = ["needle", "Intraosseous"]
Q3b = "What are the safety warnings when using the intraosseous needle?"

context_t3b = retrieve_kg_context(query_entities=Q3_WARN)
answer_t3b  = generate_answer(Q3b, context_t3b)
show_result(Q3b, answer_t3b, context_t3b, "Track 3 — KG (warning retrieval)")

# Show warnings explicitly
warnings = [r for r in context_t3b if r.get('warning')]
if warnings:
    display(Markdown("#### ⚠️ Warnings retrieved directly from KG:"))
    for w in warnings:
        display(Markdown(f"- {w['warning']}"))

---
## 4. Track 4 — Hybrid Graph + Vector
Vector search finds the most relevant chunks, then the graph expands to linked Steps, Figures, and Warnings. Best of both worlds.

In [ ]:
from retrieval.search_hybrid import retrieve_hybrid_context

def format_hybrid(raw):
    """Merge hybrid rows into unified text blocks."""
    out, seen = [], set()
    for r in raw:
        parts = []
        for key in ["chunk_text", "step_text"]:
            v = r.get(key) or ""
            if v: parts.append(v)
        if r.get("llm_caption"):  parts.append(f"Figure caption: {r['llm_caption']}")
        if r.get("ocr_text") and r["ocr_text"].lower() != "none":
            parts.append(f"Figure OCR: {r['ocr_text']}")
        combined = " | ".join(parts).strip()
        if combined and combined not in seen:
            seen.add(combined)
            out.append({"text": combined, "doc": r.get("doc", "?"), "figure_path": r.get("figure_path","")})
    return out

# A complex question requiring both text context AND figure understanding
Q4 = "What are the steps to perform CPR and when should I use the AED?"

raw_t4    = retrieve_hybrid_context(query=Q4, top_k=3)
context_t4 = format_hybrid(raw_t4)
answer_t4  = generate_answer(Q4, context_t4)
show_result(Q4, answer_t4, context_t4, "Track 4 — Hybrid Graph + Vector")

In [ ]:
# Show what the graph expansion added on top of the vector search
display(Markdown("#### What the graph expansion added:"))

with_figs    = [c for c in context_t4 if c.get("figure_path")]
without_figs = [c for c in context_t4 if not c.get("figure_path")]

display(Markdown(f"- **{len(without_figs)}** text/step chunks (from vector search)"))
display(Markdown(f"- **{len(with_figs)}** chunks enriched with figure captions (from graph expansion)"))

if with_figs:
    display(Markdown("\n**Figures linked via graph:**"))
    for c in with_figs:
        fig_p = Path("../data") / c["figure_path"]
        display(Markdown(f"`{c['figure_path']}`"))
        if fig_p.exists():
            display(IPImage(str(fig_p), width=350))
        caption_part = [p for p in c["text"].split(" | ") if "caption" in p.lower()]
        if caption_part:
            display(Markdown(f"*{caption_part[0]}*"))

---
## 5. All-Tracks Comparison
Same question through all four tracks side by side.

In [ ]:
# Pick a question where image context clearly helps
DEMO_Q = "How do I identify the correct intraosseous insertion site on the leg?"

display(Markdown(f"## Question: *{DEMO_Q}*"))
display(Markdown("---"))

# Track 1
c1  = retrieve_text_context(query=DEMO_Q, top_k=4)
a1  = generate_answer(DEMO_Q, c1)
display(Markdown("### 🔵 Track 1 — Raw Text"))
display(Markdown(a1))
print()

# Track 2
c2  = retrieve_enriched_context(query=DEMO_Q, top_k=4)
a2  = generate_answer(DEMO_Q, c2)
display(Markdown("### 🟢 Track 2 — Enriched Text (with image captions)"))
display(Markdown(a2))
print()

# Track 3
c3  = retrieve_kg_context(query_entities=["Intraosseous", "insertion", "leg"])
a3  = generate_answer(DEMO_Q, c3)
display(Markdown("### 🟠 Track 3 — KG Structured"))
display(Markdown(a3))
print()

# Track 4
c4_raw = retrieve_hybrid_context(query=DEMO_Q, top_k=3)
c4     = format_hybrid(c4_raw)
a4     = generate_answer(DEMO_Q, c4)
display(Markdown("### 🔴 Track 4 — Hybrid Graph + Vector"))
display(Markdown(a4))

# Show the relevant figure
fig_io = Path("../data/images/1.103_ALS_ALGORITHM/fig2.png")
if fig_io.exists():
    print()
    display(Markdown("**Figure referenced in procedure (only Track 2 & 4 can access this):**"))
    display(IPImage(str(fig_io), width=380))
    display(Markdown("*Figure 2 — Intraosseous Site (from 1.103 ALS ALGORITHM)*"))

---
## 6. Summary Table
Quick recap of what each track retrieved for the comparison question.

In [ ]:
import pandas as pd

summary = [
    {
        "Track": "Track 1 — Raw Text",
        "Chunks retrieved": len(c1),
        "Unique docs": len({c.get('doc') for c in c1}),
        "Has image info": "❌",
        "Answer length": len(a1.split()),
    },
    {
        "Track": "Track 2 — Enriched",
        "Chunks retrieved": len(c2),
        "Unique docs": len({c.get('doc') for c in c2}),
        "Has image info": "✅" if any('[IMAGE DESCRIPTION' in c.get('text','') for c in c2) else "❌",
        "Answer length": len(a2.split()),
    },
    {
        "Track": "Track 3 — KG",
        "Chunks retrieved": len(c3),
        "Unique docs": len({c.get('doc') for c in c3}),
        "Has image info": "✅" if any(c.get('llm_caption') for c in c3) else "❌",
        "Answer length": len(a3.split()),
    },
    {
        "Track": "Track 4 — Hybrid",
        "Chunks retrieved": len(c4),
        "Unique docs": len({c.get('doc') for c in c4}),
        "Has image info": "✅" if any(c.get('figure_path') for c in c4) else "❌",
        "Answer length": len(a4.split()),
    },
]

df = pd.DataFrame(summary).set_index("Track")
display(df)